### Unpacking self-attention

<b>Logits:</b> Raw, unnormalised scores generated by the final layer of a neural netowrk before an activation function such as Softmax or Sigmoid

In [1]:
import numpy as np

In [17]:
def softmax(x):
    ## axis = 0 for column-wise, axis = 1 for row-wise, axis = -1 for last dimension
    ## keepdims = True to maintain the same number of dimensions for broadcasting, otherwise dimension will be reduced
    shifted = x - np.max(x, axis = -1, keepdims = True) 
    exp_x = np.exp(shifted)
    return exp_x / np.sum(exp_x, axis = -1, keepdims = True)

logits = np.array([2.0, 1.0, 0.1])
print(f"logits:  {logits}")
print(f"softmax: {softmax(logits)}")
print(f"sum:     {softmax(logits).sum():.4f}")

logits:  [2.  1.  0.1]
softmax: [0.65900114 0.24243297 0.09856589]
sum:     1.0000


* Softmax only cares about the relative differences between the numbers and not the exact values. Hence, there's the shifted = x - max value within the list
* Without this, there will be overflow/ underflow, whereby either all the values are large leading to inf or all the absolute values are small leading to 0 

##### Why substract the maxmimum instead of the minimum value across x
By subtracting the maxmimum value, the exponenet of all values are guarenteed to be under 1 as e**0 = 1 when xi​−max(x)≤0. 

On the other hand, if subtracting the minimum leads to x = 2000, then e**2000 becomes an infinitely large value. Coversely, the x = -2000 may leads to undeflow to zero but is acceptable because thos eterms correspond to probabilities that are alreadyu effectively zero. (Overflow is a bigger issue than underflow in softmax)

#### Sclaed dot product attention 

In [19]:
def scaled_dot_product_attention(Q, K, V):
    d_k = Q.shape[-1]  # dimension of the key vectors
    scores = Q @ K.T / np.sqrt(d_k) # shape: (seq_len_q, seq_len_k)
    weights = softmax(scores)  # shape: (seq_len_q, seq_len_k)
    output = weights @ V  # shape: (seq_len_q, d_v)
    return output, weights

##### Self-attention class with learned projections

In [21]:
class SelfAttention:
    def __init__(self, d_model, d_k, d_v, seed = 42):
        rng = np.random.default_rng(seed)
        scale = np.sqrt(2.0 / (d_model + d_k))
        self.W_Q = rng.normal(0, scale, (d_model, d_k))
        self.W_K = rng.normal(0, scale, (d_model, d_k))
        scale_v = np.sqrt(2.0 / (d_model + d_v))
        self.W_V = rng.normal(0, scale_v, (d_model, d_v))
        self.d_k = d_k

    def forward(self, X):
        Q = X @ self.W_Q  # shape: (seq_len, d_k)
        K = X @ self.W_K  # shape: (seq_len, d_k)
        V = X @ self.W_V  # shape: (seq_len, d_v)
        output, weights = scaled_dot_product_attention(Q, K, V)
        return output, weights

In [23]:
X

array([[ 0.30471708, -1.03998411,  0.7504512 ,  0.94056472, -1.95103519,
        -1.30217951,  0.1278404 , -0.31624259],
       [-0.01680116, -0.85304393,  0.87939797,  0.77779194,  0.0660307 ,
         1.12724121,  0.46750934, -0.85929246],
       [ 0.36875078, -0.9588826 ,  0.8784503 , -0.04992591, -0.18486236,
        -0.68092954,  1.22254134, -0.15452948],
       [-0.42832782, -0.35213355,  0.53230919,  0.36544406,  0.41273261,
         0.430821  ,  2.1416476 , -0.40641502],
       [-0.51224273, -0.81377273,  0.61597942,  1.12897229, -0.11394746,
        -0.84015648, -0.82448122,  0.65059279],
       [ 0.74325417,  0.54315427, -0.66550971,  0.23216132,  0.11668581,
         0.2186886 ,  0.87142878,  0.22359555]])

In [22]:
sentence = ["The", "cat", "sat", "on", "the", "mat"]
n_tokens = len(sentence)
d_model = 8
dk = 4
dv = 4

rng = np.random.default_rng(42)
X = rng.normal(0, 1, (n_tokens, d_model))

attn = SelfAttention(d_model, dk, dv, seed=42)
output, weights = attn.forward(X)

print("Attention weights (each row: where that token looks):\n")
print(f"{'':>6}", end="")
for token in sentence:
    print(f"{token:>6}", end="")
print()

for i, token in enumerate(sentence):
    print(f"{token:>6}", end="")
    for j in range(n_tokens):
        w = weights[i][j]
        print(f"{w:6.3f}", end="")
    print()

Attention weights (each row: where that token looks):

         The   cat   sat    on   the   mat
   The 0.097 0.122 0.236 0.445 0.047 0.052
   cat 0.188 0.150 0.176 0.144 0.193 0.149
   sat 0.166 0.130 0.213 0.187 0.150 0.154
    on 0.172 0.144 0.146 0.115 0.208 0.215
   the 0.198 0.170 0.214 0.246 0.129 0.043
   mat 0.168 0.152 0.126 0.101 0.220 0.234


In [36]:
def ascii_heatmap(weights, tokens, chars=" ░▒▓█"):
    n = len(tokens)
    print(f"\n{'':>6}", end="")
    for t in tokens:
        print(f"{t:>6}", end="")
    print()

    for i in range(n):
        print(f"{tokens[i]:>6}", end="")
        for j in range(n):
            level = int(weights[i][j] * (len(chars) - 1) / weights.max())
            level = min(level, len(chars) - 1)
            print(f"{'  ' + chars[level] + '   '}", end="")
        print()

ascii_heatmap(weights, sentence)


         The   cat   sat    on   the   mat
   The        ░     ▒     █               
   cat  ░     ░     ░     ░     ░     ░   
   sat  ░     ░     ░     ░     ░     ░   
    on  ░     ░     ░     ░     ░     ░   
   the  ░     ░     ░     ▒     ░         
   mat  ░     ░     ░           ░     ▒   


In [38]:
import torch
import torch.nn as nn

d_model = 8
n_heads = 2
seq_len = 6

mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, batch_first=True)

X_torch = torch.randn(1, seq_len, d_model)

output, attn_weights = mha(X_torch, X_torch, X_torch)

print(f"Input shape:            {X_torch.shape}")
print(f"Output shape:           {output.shape}")
print(f"Attention weight shape: {attn_weights.shape}")
print(f"\nAttn weights (averaged over heads):")
print(attn_weights[0].detach().numpy().round(3))

Input shape:            torch.Size([1, 6, 8])
Output shape:           torch.Size([1, 6, 8])
Attention weight shape: torch.Size([1, 6, 6])

Attn weights (averaged over heads):
[[0.16  0.15  0.165 0.212 0.176 0.137]
 [0.207 0.121 0.138 0.232 0.17  0.133]
 [0.226 0.11  0.166 0.224 0.156 0.118]
 [0.13  0.201 0.208 0.122 0.198 0.14 ]
 [0.189 0.162 0.151 0.171 0.142 0.184]
 [0.171 0.136 0.115 0.287 0.172 0.12 ]]
